# Belle II Continuum Suppression

Binary signal-versus-continuum classification using the instructor's Belle II CSV. The private data is not committed.

## 1. Setup and reproducibility

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.config import (
    DB_PATH, RANDOM_STATE, RAW_DATA_DIR, TEST_SIZE, VALIDATION_SIZE
)
from src.data_io import (
    assign_split, audit_frame, load_raw_to_db, prepare_professor_binary
)
from src.db import read_table, table_exists, write_table

print(f"Project root: {PROJECT_ROOT}")
print(
    f"Random state: {RANDOM_STATE}; split fractions: "
    f"train={1 - VALIDATION_SIZE - TEST_SIZE:.0%}, "
    f"validation={VALIDATION_SIZE:.0%}, test={TEST_SIZE:.0%}"
)

## 2. Instructor dataset

Use `data_hep - data_hep.csv` only. Its process flags are converted to the binary target below.

The original CSV remains private and unchanged.

## 3. Locate the instructor CSV

In [ ]:
raw_csv_path = RAW_DATA_DIR / "data_hep - data_hep.csv"
if not raw_csv_path.is_file():
    raw_csv_path = None

if raw_csv_path is None:
    print("Waiting for data/raw/data_hep - data_hep.csv.")
else:
    print(f"Dataset found: {raw_csv_path.name}")

## 4. Store the untouched CSV in SQLite

In [ ]:
raw_events = None
if raw_csv_path is not None:
    if table_exists("raw_events", DB_PATH):
        raw_events = read_table("raw_events", DB_PATH)
        print("Read the existing immutable raw_events table.")
    else:
        raw_events = load_raw_to_db(raw_csv_path, DB_PATH)
        print("Wrote the immutable raw_events table.")
    print(f"Loaded {len(raw_events):,} rows and {raw_events.shape[1]} columns.")
else:
    print("Ingestion skipped until exactly one source CSV is available.")

## 5. Audit the raw data

In [ ]:
raw_audit = audit_frame(raw_events) if raw_events is not None else None
if raw_audit is not None:
    print(f"Rows: {raw_audit['rows']:,}; columns: {raw_audit['columns']}")
    print(f"Duplicate rows: {raw_audit['duplicate_rows']:,}")
    print(f"Constant columns: {raw_audit['constant_columns']}")
    print(f"Six-way type counts: {raw_events['type'].value_counts().sort_index().to_dict()}")
    print(f"Missing cells: {sum(raw_audit['missing_counts'].values()):,}")
    print(f"Non-finite cells: {sum(raw_audit['non_finite_counts'].values()):,}")
    display(raw_events.head())
else:
    print("Audit skipped because no dataset has been loaded.")

## 6. Prepare the binary label

`type` 0/1 → signal (`target=1`); 2/3/4/5 → background (`target=0`). Exclude `type` and the exported row number from model inputs.

In [ ]:
prepared_events = prepare_professor_binary(raw_events) if raw_events is not None else None
if prepared_events is not None:
    print(f"Rows retained: {len(prepared_events):,}/{len(raw_events):,}")
    print("Excluded from prepared data: Unnamed: 0 and six-way type")
    display(prepared_events['target'].value_counts().sort_index().rename('rows'))
else:
    print("Binary preparation waits for the professor CSV.")

## 7. Store the 70/15/15 train/validation/test split

In [ ]:
events_with_split = None
if prepared_events is not None:
    if table_exists("cleaned_events", DB_PATH):
        events_with_split = read_table("cleaned_events", DB_PATH)
        print("Read persisted cleaned_events; did not re-split.")
    else:
        events_with_split = assign_split(
            prepared_events, "target",
            validation_size=VALIDATION_SIZE, test_size=TEST_SIZE,
        )
        write_table(events_with_split, "cleaned_events", db_path=DB_PATH)
        print("Wrote cleaned_events with one persisted split.")
    display(events_with_split.groupby(["split", "target"]).size().rename("rows"))
else:
    print("Split assignment waits for professor data.")

## Next: exploratory data analysis